In [61]:
import pandas as pd
import numpy as np
import os
import warnings
import argparse
import skimpy as sk
import mlflow
import mlflow.sklearn
import seaborn as sns
import matplotlib.pyplot as plt
import json
from pathlib import Path
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols, logit, gee
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families import Binomial
from patsy import dmatrix
from mlflow.models import infer_signature

from sklearn.preprocessing import *
from sklearn.model_selection import *
from imblearn.pipeline import Pipeline
from sklearn.metrics import *
from sklearn.linear_model import *
from sklearn.ensemble import *
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.utils.validation import check_is_fitted
from sklearn.base import *
from xgboost import *
from lightgbm import *
from catboost import *

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("data/LUMA1_assessment_data.csv")

In [3]:
sk.skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 240    │ │ float64     │ 14    │                                                          │
│ │ Number of columns │ 22     │ │ string      │ 4     │                                                          │
│ └───────────────────┴────────┘ │ int64       │ 4     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┓  │
│ ┃ column           ┃ NA ┃ NA %             ┃ mean  ┃ sd     ┃ p0    ┃ p25   ┃ p50   ┃ p75   ┃ p100  ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━┩  │
│ │   assay_batch    │  0 │                0 │ 1.988 │ 0.8207 │     1 │     1 │     2 │     3 │     3 │ █  █ █ │  │
│ │ chamber_position │  0 │                0 │   3.5 │  1.711 │     1 │     2 │   3.5 │     5 │     6 │ ██████ │  │
│ │   temperature    │  0 │                0 │ 25.05 │ 0.3948 │ 24.19 │ 24.77 │ 25.02 │  25.4 │ 25.74 │ ▂▃█▇▅▅ │  │
│ │     humidity     │  0 │                0 │ 60.02 │  1.634 │  56.7 │  58.9 │ 60.15 │  61.2 │  63.8 │ ▃▄▅█▂▂ │  │
│ │    body_mass     │  0 │                0 │ 49.85 │  5.047 │  35.9 │ 46.84 │ 49.63 │ 53.16 │ 63.49 │ ▁▃█▇▄▁ │  │
│ │ baseline_activit │  3 │             1.25 │ 49.76 │  10.44 │ 22.53 │ 42.28 │ 49.07 │ 55.74 │ 84.37 │ ▁▄█▄▁  │  │
│ │        y         │    │                  │       │        │       │       │       │       │       │        │  │
│ │ baseline_feeding │  0 │                0 │ 29.46 │  5.921 │ 10.93 │ 25.76 │ 29.09 │ 33.57 │ 48.58 │  ▂█▆▃  │  │
│ │   activity_24h   │  0 │                0 │ 42.97 │  10.72 │ 17.11 │ 35.39 │ 42.56 │ 49.99 │  74.9 │ ▂▅██▂▁ │  │
│ │   feeding_24h    │ 11 │ 4.58333333333333 │ 25.64 │  6.194 │  9.27 │ 21.41 │ 25.74 │ 29.96 │ 41.75 │ ▁▄██▄▁ │  │
│ │                  │    │                3 │       │        │       │       │       │       │       │        │  │
│ │ response_latency │  0 │                0 │ 20.13 │  4.753 │  5.99 │ 17.06 │ 19.98 │ 23.22 │ 32.43 │  ▃▆█▄▁ │  │
│ │   oxygen_rate    │ 15 │             6.25 │ 1.835 │ 0.3541 │ 0.656 │ 1.604 │ 1.853 │ 2.074 │  2.68 │  ▂▇█▇▂ │  │
│ │ molecular_signal │  0 │                0 │  5.57 │  1.187 │ 1.931 │ 4.809 │ 5.518 │ 6.405 │ 8.679 │  ▂██▅▁ │  │
│ │        _A        │    │                  │       │        │       │       │       │       │       │        │  │
│ │ molecular_signal │ 32 │ 13.3333333333333 │ 2.634 │ 0.6452 │ 1.564 │ 2.187 │ 2.602 │ 3.018 │ 4.525 │ ▆▇█▄▂▁ │  │
│ │        _B        │    │               34 │       │        │       │       │       │       │       │        │  │
│ │ fluorescence_sig │  0 │                0 │ 66.07 │  10.71 │ 29.75 │ 58.61 │ 65.03 │  73.8 │   100 │  ▂█▇▃▁ │  │
│ │       nal        │    │                  │       │        │       │       │       │       │       │        │  │
│ │ sensor_variabili │  0 │                0 │ 1.295 │  0.264 │  1.02 │ 1.189 │ 1.252 │ 1.369 │ 4.757 │   █    │  │
│ │        ty        │    │                  │       │        │       │       │       │       │       │        │  │
│ │ processing_delay │  0 │                0 │ 36.48 │  

## Causal Assumptions
- chamber_position -> measurement_order -> processing_delay -> activity_24h -> treatment
- sensor_variability -> response_latency -> processing_delay -> treatment
- molecular_signal_A <-> molecular_signal_B -> treatment

I'm assuming this is an IoT Analytics Problem thats specific for health

## EDA

In [4]:
df.columns

Index(['individual_id', 'chamber_id', 'source_group', 'treatment',
       'assay_batch', 'chamber_position', 'temperature', 'humidity',
       'body_mass', 'baseline_activity', 'baseline_feeding', 'activity_24h',
       'feeding_24h', 'response_latency', 'oxygen_rate', 'molecular_signal_A',
       'molecular_signal_B', 'fluorescence_signal', 'sensor_variability',
       'processing_delay', 'measurement_order', 'recovered_48h'],
      dtype='str')

In [10]:
INPUT = "LUMA1_assessment_data.csv"
OUT = "LUMA1_results"

TABLES = os.path.join(OUT, "tables")
FIGURES = os.path.join(OUT, "figures")
REPORTS = os.path.join(OUT, "reports")

for directory in [TABLES, FIGURES, REPORTS]:
    os.makedirs(directory, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["savefig.dpi"] = 180


print("=" * 65)
print("LUMA-1 SCIENTIFIC EDA")
print("=" * 65)
print("Shape:", df.shape)

print("=" * 65)
print("LUMA-1 SCIENTIFIC EDA")
print("=" * 65)
print("Shape:", df.shape)

# ============================================================
# VARIABLE DEFINITIONS
# ============================================================

id_cols = [
    "individual_id", "chamber_id"
]

categorical_cols = [
    "source_group", "treatment", "assay_batch"
]

continuous_cols = [
    "temperature", "humidity", "chamber_position",
    "body_mass", "baseline_activity", "baseline_feeding",
    "activity_24h", "feeding_24h", "response_latency",
    "oxygen_rate", "molecular_signal_A", "molecular_signal_B",
    "fluorescence_signal", "sensor_variability",
    "processing_delay", "measurement_order"
]

baseline_cols = [
    "body_mass", "baseline_activity", "baseline_feeding"
]

post_cols = [
    "activity_24h", "feeding_24h", "response_latency",
    "oxygen_rate", "molecular_signal_A",
    "molecular_signal_B", "fluorescence_signal"
]

mechanism_groups = {
    "operational": [
        "chamber_position", "measurement_order",
        "processing_delay", "assay_batch"
    ],
    "instrumentation": [
        "sensor_variability", "response_latency",
        "processing_delay"
    ],
    "physiological": [
        "baseline_activity", "activity_24h",
        "baseline_feeding", "feeding_24h",
        "oxygen_rate"
    ],
    "molecular": [
        "molecular_signal_A", "molecular_signal_B",
        "fluorescence_signal"
    ]
}

# ============================================================
# 01. DATA AUDIT
# ============================================================

print("\n[01] DATA AUDIT")

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "unique": df.nunique()
})

audit.to_csv(f"{TABLES}/01_data_audit.csv")

print(audit)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate individual IDs:", df["individual_id"].duplicated().sum())

print("\nTreatment distribution:")
print(df.groupby("treatment").agg(
    individuals=("individual_id", "count"),
    chambers=("chamber_id", "nunique"),
    recovery_rate=("recovered_48h", "mean")
))

print("\nChamber structure:")
chamber_audit = df.groupby("chamber_id").agg(
    individuals=("individual_id", "count"),
    treatment_count=("treatment", "nunique"),
    treatment=("treatment", "first"),
    recovery_rate=("recovered_48h", "mean")
)

chamber_audit.to_csv(f"{TABLES}/01_chamber_audit.csv")
print(chamber_audit)

print("\nTreatment allocation across chambers:")
print(pd.crosstab(
    df["chamber_id"],
    df["treatment"]
))

# Check chamber-level consistency
if (chamber_audit["treatment_count"] > 1).any():
    print("WARNING: Some chambers contain multiple treatments.")

if not (chamber_audit["individuals"] == 8).all():
    print("WARNING: Not all chambers contain exactly 8 individuals.")

# ============================================================
# 02. QUARTILE DISTRIBUTION
# ============================================================

print("\n[02] QUARTILE DISTRIBUTION")

numeric = df[continuous_cols].apply(pd.to_numeric, errors="coerce")

quartiles = numeric.describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).T

quartiles["IQR"] = quartiles["75%"] - quartiles["25%"]
quartiles["skewness"] = numeric.skew()
quartiles["kurtosis"] = numeric.kurtosis()

quartiles.to_csv(f"{TABLES}/02_quartile_distribution.csv")

print(quartiles)

# Histograms and KDE
for col in continuous_cols:
    fig, ax = plt.subplots()

    sns.histplot(
        data=df,
        x=col,
        kde=True,
        bins=20,
        ax=ax
    )

    ax.set_title(f"Distribution: {col}")
    plt.tight_layout()
    plt.savefig(f"{FIGURES}/distribution_{col}.png")
    plt.close()

# Boxplots by treatment
for col in continuous_cols:
    fig, ax = plt.subplots()

    sns.boxplot(
        data=df,
        x="treatment",
        y=col,
        hue="treatment",
        legend=False,
        ax=ax
    )

    ax.set_title(f"{col} by treatment")
    plt.tight_layout()
    plt.savefig(f"{FIGURES}/boxplot_{col}_treatment.png")
    plt.close()

# ECDF distributions
for col in continuous_cols:
    fig, ax = plt.subplots()

    sns.ecdfplot(
        data=df,
        x=col,
        hue="treatment",
        ax=ax
    )

    ax.set_title(f"ECDF: {col}")
    plt.tight_layout()
    plt.savefig(f"{FIGURES}/ecdf_{col}.png")
    plt.close()

# ============================================================
# 03. OUTLIER DETECTION
# ============================================================

print("\n[03] OUTLIER DETECTION")

outlier_results = []

for col in continuous_cols:

    x = numeric[col].dropna()

    if len(x) == 0:
        continue

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    iqr_mask = (x < lower) | (x > upper)

    median = x.median()
    mad = np.median(np.abs(x - median))

    if mad > 0:
        modified_z = 0.6745 * (x - median) / mad
        mad_mask = modified_z.abs() > 3.5
    else:
        modified_z = pd.Series(np.nan, index=x.index)
        mad_mask = pd.Series(False, index=x.index)

    outlier_results.append({
        "variable": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_fence": lower,
        "upper_fence": upper,
        "IQR_outliers": int(iqr_mask.sum()),
        "MAD_outliers": int(mad_mask.sum()),
        "IQR_outlier_pct": 100 * iqr_mask.mean()
    })

outliers = pd.DataFrame(outlier_results)
outliers.to_csv(f"{TABLES}/03_outlier_summary.csv", index=False)

print(outliers)

# Save individual outlier observations
outlier_records = []

for col in continuous_cols:

    x = numeric[col].dropna()

    if len(x) == 0:
        continue

    q1 = x.quantile(.25)
    q3 = x.quantile(.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (x < lower) | (x > upper)

    for idx in x[mask].index:
        outlier_records.append({
            "row_index": idx,
            "individual_id": df.loc[idx, "individual_id"],
            "chamber_id": df.loc[idx, "chamber_id"],
            "treatment": df.loc[idx, "treatment"],
            "variable": col,
            "value": x.loc[idx],
            "lower_fence": lower,
            "upper_fence": upper
        })

pd.DataFrame(outlier_records).to_csv(
    f"{TABLES}/03_outlier_records.csv",
    index=False
)

# ============================================================
# 04. CORRELATION AND VIF
# ============================================================

print("\n[04] CORRELATION AND VIF")

corr = numeric.corr(method="spearman")

corr.to_csv(f"{TABLES}/04_spearman_correlation.csv")

fig, ax = plt.subplots(figsize=(15, 12))

sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
    ax=ax
)

ax.set_title("Spearman Correlation Matrix")
plt.tight_layout()
plt.savefig(f"{FIGURES}/04_correlation_heatmap.png")
plt.close()

# VIF for baseline/environmental candidate covariates.
# Categorical variables are one-hot encoded.
# This is a diagnostic candidate set, not a causal adjustment set.

vif_data = df.copy()

categorical_cols = [
    "source_group",
    "treatment",
    "assay_batch"
]

for col in categorical_cols:
    vif_data[col] = vif_data[col].astype(object)
    vif_data[col] = vif_data[col].where(
        vif_data[col].notna(), np.nan
    )

vif_formula = (
    "body_mass + baseline_activity + baseline_feeding + "
    "temperature + humidity + "
    "C(source_group) + C(treatment) + C(assay_batch)"
)

vif_design = dmatrix(
    vif_formula,
    vif_data,
    return_type="dataframe",
    NA_action="drop"
)

# Remove intercept and constant columns.
vif_design = vif_design.drop(
    columns=["Intercept"],
    errors="ignore"
)

vif_design = vif_design.loc[
    :, vif_design.nunique() > 1
]

# Remove linearly dependent columns before calculating VIF.
# This prevents singular design matrices from producing misleading results.
X = vif_design.astype(float)

independent_cols = []
current_rank = 0

for col in X.columns:
    candidate = X[independent_cols + [col]].to_numpy()
    new_rank = np.linalg.matrix_rank(candidate)

    if new_rank > current_rank:
        independent_cols.append(col)
        current_rank = new_rank

X = X[independent_cols]

# Calculate VIF.
if X.shape[1] >= 2:
    vif_results = pd.DataFrame({
        "Variable": X.columns,
        "VIF": [
            variance_inflation_factor(X.to_numpy(), i)
            for i in range(X.shape[1])
        ]
    }).sort_values("VIF", ascending=False)

    display(vif_results)
else:
    print("Insufficient independent predictors for VIF.")


# ============================================================
# 05. TREATMENT EFFECTS AND SENSITIVITY
# ============================================================

print("\n[05] TREATMENT EFFECTS")

recovery = df.groupby("treatment")["recovered_48h"].agg(
    recovered="sum",
    total="count",
    recovery_rate="mean"
)

recovery["non_recovered"] = (
    recovery["total"] - recovery["recovered"]
)

recovery.to_csv(f"{TABLES}/05_recovery_rates.csv")

print(recovery)

# Wilson confidence intervals
from statsmodels.stats.proportion import proportion_confint

ci_results = []

for treatment, row in recovery.iterrows():

    low, high = proportion_confint(
        int(row["recovered"]),
        int(row["total"]),
        method="wilson"
    )

    ci_results.append({
        "treatment": treatment,
        "rate": row["recovery_rate"],
        "lower_95": low,
        "upper_95": high
    })

ci_df = pd.DataFrame(ci_results)
ci_df.to_csv(f"{TABLES}/05_recovery_confidence_intervals.csv", index=False)

# Chamber-level bootstrap.
# Resample chambers within each treatment.
# Retain all individuals belonging to each sampled chamber.

rng = np.random.default_rng(42)

def chamber_bootstrap(data, treatment_a, treatment_b, n_boot=5000):

    grouped = {
        t: [
            g["recovered_48h"].to_numpy()
            for _, g in data[data["treatment"] == t].groupby("chamber_id")
        ]
        for t in [treatment_a, treatment_b]
    }

    results = []

    for _ in range(n_boot):

        rates = {}

        for treatment in [treatment_a, treatment_b]:

            chambers = grouped[treatment]

            sampled = rng.integers(
                0, len(chambers), size=len(chambers)
            )

            values = np.concatenate([
                chambers[i] for i in sampled
            ])

            rates[treatment] = values.mean()

        results.append(
            rates[treatment_b] - rates[treatment_a]
        )

    return np.asarray(results)


bootstrap_results = []

treatments = sorted(df["treatment"].dropna().unique())

for i, a in enumerate(treatments):
    for b in treatments[i+1:]:

        boot = chamber_bootstrap(df, a, b)

        bootstrap_results.append({
            "comparison": f"{b} - {a}",
            "mean_difference": boot.mean(),
            "lower_95": np.quantile(boot, .025),
            "upper_95": np.quantile(boot, .975)
        })

bootstrap_df = pd.DataFrame(bootstrap_results)
bootstrap_df.to_csv(
    f"{TABLES}/05_cluster_bootstrap.csv",
    index=False
)

print("\nChamber bootstrap:")
print(bootstrap_df)

# Leave-one-chamber-out analysis
loo_results = []

for chamber in df["chamber_id"].unique():

    subset = df[df["chamber_id"] != chamber]

    for treatment in treatments:

        group = subset[subset["treatment"] == treatment]

        loo_results.append({
            "excluded_chamber": chamber,
            "treatment": treatment,
            "n": len(group),
            "recovery_rate": group["recovered_48h"].mean()
        })

loo = pd.DataFrame(loo_results)
loo.to_csv(f"{TABLES}/05_leave_one_chamber_out.csv", index=False)

# GEE: population-averaged recovery model.
# Cluster = chamber.
# Use only pre-treatment covariates as adjustments.

gee_data = df.dropna(subset=[
    "recovered_48h",
    "treatment",
    "chamber_id",
    "body_mass",
    "baseline_activity",
    "baseline_feeding"
]).copy()

gee_formula = (
    "recovered_48h ~ C(treatment) + body_mass + "
    "baseline_activity + baseline_feeding"
)

try:

    gee_model = gee(
        gee_formula,
        groups="chamber_id",
        data=gee_data,
        family=Binomial(),
        cov_struct=Exchangeable()
    ).fit()

    with open(f"{REPORTS}/05_gee_model.txt", "w") as f:
        f.write(gee_model.summary().as_text())

    print("\nGEE MODEL")
    print(gee_model.summary())

except Exception as e:
    print("GEE model failed:", e)

# ============================================================
# 06. MECHANISM ANALYSIS
# ============================================================

print("\n[06] MECHANISM ANALYSIS")

mechanism_pairs = [
    ("chamber_position", "measurement_order"),
    ("measurement_order", "processing_delay"),
    ("sensor_variability", "response_latency"),
    ("processing_delay", "activity_24h"),
    ("processing_delay", "molecular_signal_A"),
    ("molecular_signal_A", "molecular_signal_B"),
    ("activity_24h", "recovered_48h"),
    ("feeding_24h", "recovered_48h"),
    ("oxygen_rate", "recovered_48h")
]

mechanism_results = []

for x, y in mechanism_pairs:

    subset = df[[x, y]].dropna()

    if len(subset) < 3:
        continue

    rho, p = stats.spearmanr(
        subset[x], subset[y]
    )

    mechanism_results.append({
        "variable_x": x,
        "variable_y": y,
        "n": len(subset),
        "spearman_rho": rho,
        "p_value": p
    })

mechanism_df = pd.DataFrame(mechanism_results)

mechanism_df.to_csv(
    f"{TABLES}/06_mechanism_correlations.csv",
    index=False
)

print(mechanism_df)

# Visualize proposed pathways
for x, y in mechanism_pairs:

    fig, ax = plt.subplots()

    if df[y].nunique() <= 2:

        sns.boxplot(
            data=df,
            x=y,
            y=x,
            hue=y,
            legend=False,
            ax=ax
        )

    else:

        sns.regplot(
            data=df,
            x=x,
            y=y,
            lowess=True,
            scatter_kws={"alpha": .6},
            line_kws={"color": "red"},
            ax=ax
        )

    ax.set_title(f"Mechanism: {x} vs {y}")
    plt.tight_layout()
    plt.savefig(
        f"{FIGURES}/06_mechanism_{x}_{y}.png"
    )
    plt.close()

# Baseline-to-24-hour changes
change_pairs = [
    ("baseline_activity", "activity_24h"),
    ("baseline_feeding", "feeding_24h")
]

for baseline, followup in change_pairs:

    df[f"change_{followup}"] = (
        df[followup] - df[baseline]
    )

    fig, ax = plt.subplots()

    sns.boxplot(
        data=df,
        x="treatment",
        y=f"change_{followup}",
        hue="treatment",
        legend=False,
        ax=ax
    )

    ax.axhline(0, linestyle="--", color="black")
    ax.set_title(f"Change from baseline: {followup}")

    plt.tight_layout()
    plt.savefig(
        f"{FIGURES}/06_change_{followup}.png"
    )
    plt.close()

# Molecular signal relationship
fig, ax = plt.subplots()

sns.scatterplot(
    data=df,
    x="molecular_signal_A",
    y="molecular_signal_B",
    hue="treatment",
    style="assay_batch",
    ax=ax
)

ax.set_title("Molecular Signal Relationship")
plt.tight_layout()
plt.savefig(f"{FIGURES}/06_molecular_relationship.png")
plt.close()

# ============================================================
# 07. JUMP-PROCESS / DISCONTINUITY EVALUATION
# ============================================================

print("\n[07] JUMP-PROCESS EVALUATION")

# Measurement order is an ordering variable, not necessarily
# equally spaced physical time. Analyze within assay batches.

jump_results = []

jump_variables = [
    "processing_delay",
    "sensor_variability",
    "molecular_signal_A",
    "molecular_signal_B",
    "fluorescence_signal",
    "response_latency"
]

ordered = df.dropna(
    subset=["measurement_order", "assay_batch"]
).copy()

for batch, group in ordered.groupby("assay_batch"):

    group = group.sort_values("measurement_order")

    for variable in jump_variables:

        temp = group[
            ["measurement_order", variable]
        ].dropna()

        if len(temp) < 4:
            continue

        values = temp[variable].to_numpy()
        orders = temp["measurement_order"].to_numpy()

        differences = np.diff(values)
        median = np.median(differences)
        mad = np.median(np.abs(differences - median))

        if mad > 0:
            robust_jump = (
                0.6745 * (differences - median) / mad
            )
        else:
            robust_jump = np.zeros_like(differences)

        for j, difference in enumerate(differences):

            jump_results.append({
                "assay_batch": batch,
                "variable": variable,
                "order_before": orders[j],
                "order_after": orders[j+1],
                "difference": difference,
                "absolute_difference": abs(difference),
                "robust_difference_z": robust_jump[j],
                "potential_jump": abs(robust_jump[j]) > 3.5
            })

jump_df = pd.DataFrame(jump_results)

jump_df.to_csv(
    f"{TABLES}/07_jump_diagnostics.csv",
    index=False
)

if not jump_df.empty:

    jump_summary = jump_df.groupby(
        ["assay_batch", "variable"]
    ).agg(
        n_differences=("difference", "count"),
        median_absolute_difference=("absolute_difference", "median"),
        max_absolute_difference=("absolute_difference", "max"),
        potential_jumps=("potential_jump", "sum")
    ).reset_index()

    jump_summary.to_csv(
        f"{TABLES}/07_jump_summary.csv",
        index=False
    )

    print(jump_summary)

# Plot measurements against measurement order
for variable in jump_variables:

    fig, ax = plt.subplots()

    sns.scatterplot(
        data=df,
        x="measurement_order",
        y=variable,
        hue="assay_batch",
        alpha=.7,
        ax=ax
    )

    ax.set_title(f"{variable} vs measurement order")
    plt.tight_layout()
    plt.savefig(
        f"{FIGURES}/07_order_{variable}.png"
    )
    plt.close()

# Processing delay by measurement order
fig, ax = plt.subplots()

sns.regplot(
    data=df,
    x="measurement_order",
    y="processing_delay",
    lowess=True,
    scatter_kws={"alpha": .6},
    line_kws={"color": "red"},
    ax=ax
)

ax.set_title("Processing Delay vs Measurement Order")
plt.tight_layout()
plt.savefig(f"{FIGURES}/07_processing_order.png")
plt.close()

# Chamber position vs measurement order
fig, ax = plt.subplots()

sns.boxplot(
    data=df,
    x="chamber_position",
    y="measurement_order",
    hue="chamber_position",
    legend=False,
    ax=ax
)

ax.set_title("Measurement Order by Chamber Position")
plt.tight_layout()
plt.savefig(f"{FIGURES}/07_position_order.png")
plt.close()

# ============================================================
# 08. CHAMBER-LEVEL DIAGNOSTICS
# ============================================================

print("\n[08] CHAMBER-LEVEL DIAGNOSTICS")

chamber_metrics = [
    "temperature",
    "humidity",
    "activity_24h",
    "feeding_24h",
    "oxygen_rate",
    "response_latency",
    "processing_delay",
    "sensor_variability",
    "molecular_signal_A",
    "molecular_signal_B",
    "fluorescence_signal"
]

chamber_means = df.groupby(
    ["chamber_id", "treatment"]
)[chamber_metrics + ["recovered_48h"]].mean().reset_index()

chamber_means.to_csv(
    f"{TABLES}/08_chamber_means.csv",
    index=False
)

for variable in chamber_metrics + ["recovered_48h"]:

    fig, ax = plt.subplots()

    sns.stripplot(
        data=chamber_means,
        x="treatment",
        y=variable,
        hue="treatment",
        legend=False,
        jitter=.15,
        size=8,
        ax=ax
    )

    ax.set_title(f"Chamber-level {variable}")
    plt.tight_layout()
    plt.savefig(
        f"{FIGURES}/08_chamber_{variable}.png"
    )
    plt.close()

# ============================================================
# 09. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("LUMA-1 EDA COMPLETED")
print("=" * 65)

print("Tables:", TABLES)
print("Figures:", FIGURES)
print("Reports:", REPORTS)

print("\nPrimary outputs:")
print("01_data_audit.csv")
print("01_chamber_audit.csv")
print("02_quartile_distribution.csv")
print("03_outlier_summary.csv")
print("03_outlier_records.csv")
print("04_spearman_correlation.csv")
print("04_vif_continuous.csv")
print("04_vif_full_design.csv")
print("05_recovery_rates.csv")
print("05_recovery_confidence_intervals.csv")
print("05_cluster_bootstrap.csv")
print("05_leave_one_chamber_out.csv")
print("06_mechanism_correlations.csv")
print("07_jump_diagnostics.csv")
print("08_chamber_means.csv")

print("\nRemember: diagnostic associations are not causal proof.")

LUMA-1 SCIENTIFIC EDA
Shape: (240, 22)
LUMA-1 SCIENTIFIC EDA
Shape: (240, 22)

[01] DATA AUDIT
                       dtype  missing  missing_pct  unique
individual_id            str        0     0.000000     240
chamber_id               str        0     0.000000      30
source_group             str        0     0.000000       4
treatment                str        0     0.000000       3
assay_batch            int64        0     0.000000       3
chamber_position       int64        0     0.000000       6
temperature          float64        0     0.000000      28
humidity             float64        0     0.000000      25
body_mass            float64        0     0.000000     230
baseline_activity    float64        3     1.250000     228
baseline_feeding     float64        0     0.000000     227
activity_24h         float64        0     0.000000     234
feeding_24h          float64       11     4.583333     210
response_latency     float64        0     0.000000     221
oxygen_rate         

,Variable,VIF
0,C(source_group)[T.S2],1.822001
8,baseline_activity,1.811873
1,C(source_group)[T.S3],1.766061
9,baseline_feeding,1.734650
2,C(source_group)[T.S4],1.729089
4,C(treatment)[T.C],1.543548
3,C(treatment)[T.B],1.467539
6,C(assay_batch)[T.3],1.402459
7,body_mass,1.373783
5,C(assay_batch)[T.2],1.349408



[05] TREATMENT EFFECTS
           recovered  total  recovery_rate  non_recovered
treatment                                                
A                 33     80         0.4125             47
B                 45     80         0.5625             35
C                 36     80         0.4500             44

Chamber bootstrap:
  comparison  mean_difference  lower_95  upper_95
0      B - A         0.148043   -0.0125    0.3125
1      C - A         0.038005   -0.1000    0.1750
2      C - B        -0.111918   -0.2750    0.0500

GEE MODEL
                               GEE Regression Results                              
Dep. Variable:               recovered_48h   No. Observations:                  237
Model:                                 GEE   No. clusters:                       30
Method:                        Generalized   Min. cluster size:                   7
                      Estimating Equations   Max. cluster size:                   8
Family:                           B

## Feature Engineering

In [11]:
def engineer_luma_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Harmonize LUMA-1 variables into the cross-study schema.

    Preserves original columns and adds canonical fields.
    Does not impute, encode, scale, or remove observations.
    """

    required = [
        "individual_id",
        "chamber_id",
        "treatment",
        "temperature",
        "humidity",
        "body_mass",
        "baseline_activity",
        "baseline_feeding",
        "activity_24h",
        "feeding_24h",
        "response_latency",
        "oxygen_rate",
        "molecular_signal_A",
        "molecular_signal_B",
        "fluorescence_signal",
        "sensor_variability",
        "processing_delay",
        "measurement_order",
        "recovered_48h",
    ]

    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"Missing required LUMA-1 columns: {missing}")

    out = df.copy()

    # --------------------------------------------------------
    # CANONICAL IDENTIFIERS
    # --------------------------------------------------------

    out["study_id"] = "LUMA-1"
    out["organism"] = "LUMA-1 study organism"
    out["record_id"] = out["individual_id"].astype(str)

    # Preserve chamber as the experimental cluster.
    out["experimental_unit_id"] = out["chamber_id"].astype(str)

    # --------------------------------------------------------
    # HARMONIZED BIOLOGICAL FEATURES
    # --------------------------------------------------------

    out["recovery_temperature_c"] = pd.to_numeric(
        out["temperature"], errors="coerce"
    )

    out["body_mass"] = pd.to_numeric(
        out["body_mass"], errors="coerce"
    )

    out["treatment_group"] = out["treatment"].astype("category")

    # LUMA-specific baseline measurements.
    out["baseline_activity"] = pd.to_numeric(
        out["baseline_activity"], errors="coerce"
    )

    out["baseline_feeding"] = pd.to_numeric(
        out["baseline_feeding"], errors="coerce"
    )

    # LUMA-specific post-treatment measurements.
    out["activity_24h"] = pd.to_numeric(
        out["activity_24h"], errors="coerce"
    )

    out["feeding_24h"] = pd.to_numeric(
        out["feeding_24h"], errors="coerce"
    )

    out["response_latency"] = pd.to_numeric(
        out["response_latency"], errors="coerce"
    )

    out["oxygen_rate"] = pd.to_numeric(
        out["oxygen_rate"], errors="coerce"
    )

    # Molecular and sensor measurements.
    for col in [
        "molecular_signal_A",
        "molecular_signal_B",
        "fluorescence_signal",
        "sensor_variability",
    ]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Operational / measurement variables.
    out["processing_delay"] = pd.to_numeric(
        out["processing_delay"], errors="coerce"
    )

    out["measurement_order"] = pd.to_numeric(
        out["measurement_order"], errors="coerce"
    )

    out["humidity_pct"] = pd.to_numeric(
        out["humidity"], errors="coerce"
    )

    # --------------------------------------------------------
    # OUTCOME
    # --------------------------------------------------------

    out["recovery_state_binary"] = pd.to_numeric(
        out["recovered_48h"], errors="coerce"
    )

    if not out["recovery_state_binary"].dropna().isin([0, 1]).all():
        raise ValueError(
            "recovered_48h must contain only binary values 0 and 1."
        )

    # No continuous recovery_rate is created:
    # LUMA-1 does not directly measure one.
    
    # --------------------------------------------------------
    # MISSINGNESS INDICATORS
    # --------------------------------------------------------

    missingness_features = [
        "baseline_activity",
        "baseline_feeding",
        "activity_24h",
        "feeding_24h",
        "response_latency",
        "oxygen_rate",
        "molecular_signal_A",
        "molecular_signal_B",
        "fluorescence_signal",
        "sensor_variability",
    ]

    for col in missingness_features:
        out[f"{col}_missing"] = out[col].isna().astype("int8")

    return out


In [45]:
JUMP_SOURCE_FEATURES = [
    "processing_delay",
    "sensor_variability",
    "molecular_signal_A",
    "molecular_signal_B",
    "fluorescence_signal",
    "response_latency",
]

JUMP_FEATURES = [f"{col}_jump" for col in JUMP_SOURCE_FEATURES]


def engineer_jump_features(df):
    out = pd.DataFrame(index=df.index)

    for col in JUMP_SOURCE_FEATURES:
        values = pd.to_numeric(df[col], errors="coerce")

        median = values.median()
        mad = (values - median).abs().median()

        if pd.isna(mad) or mad == 0:
            out[f"{col}_jump"] = 0
            continue

        robust_z = 0.6745 * (values - median) / mad
        out[f"{col}_jump"] = (robust_z.abs() > 3.5).astype("int8")

    return out

In [12]:
base = df.copy()
base = engineer_luma_features(base)

In [16]:
LUMA_BASELINE_FEATURES = [
    "treatment_group",
    "recovery_temperature_c",
    "humidity_pct",
    "body_mass",
    "baseline_activity",
    "baseline_feeding",
]

LUMA_CLUSTER = "experimental_unit_id"
LUMA_TARGET = "recovery_state_binary"

X_base = base[LUMA_BASELINE_FEATURES].copy()
y_base = base[LUMA_TARGET].astype(int)
groups = base[LUMA_CLUSTER]

# Group-aware train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_base, y_base, groups=groups)
)

X_base_train = X_base.iloc[train_idx]
X_base_test = X_base.iloc[test_idx]

y_base_train = y_base.iloc[train_idx]
y_base_test = y_base.iloc[test_idx]


In [18]:
LUMA_PREDICTION_FEATURES = [
    "treatment_group",
    "recovery_temperature_c",
    "humidity_pct",
    "body_mass",
    "baseline_activity",
    "baseline_feeding",
    "activity_24h",
    "feeding_24h",
    "response_latency",
    "oxygen_rate",
    "molecular_signal_A",
    "molecular_signal_B",
    "fluorescence_signal",
    "sensor_variability",
    "processing_delay",
    "measurement_order",
]

X_pred = base[LUMA_PREDICTION_FEATURES].copy()
y_pred = base[LUMA_TARGET].astype(int)
groups_pred = base[LUMA_CLUSTER]

# Group-aware train/test split
train_idx, test_idx = next(
    GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    ).split(X_pred, y_pred, groups=groups_pred)
)

X_pred_train = X_pred.iloc[train_idx]
X_pred_test = X_pred.iloc[test_idx]

y_pred_train = y_pred.iloc[train_idx]
y_pred_test = y_pred.iloc[test_idx]

In [34]:
#Preprocessor stage
numeric_features = [
    c for c in LUMA_PREDICTION_FEATURES
    if c != "treatment_group"
]

categorical_features = ["treatment_group"]

# Logistic regression preprocessing
numeric_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Tree model preprocessing
numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("transformer", QuantileTransformer(
        n_quantiles=100,
        output_distribution="normal",
        subsample=10000,
        random_state=42
    ))
])

categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("numeric", numeric_lr, numeric_features),
    ("categorical", categorical, categorical_features)
])

preprocessor_tree = ColumnTransformer([
    ("numeric", numeric_tree, numeric_features),
    ("categorical", categorical, categorical_features)
])

In [37]:
lr = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("model", LogisticRegression(
        C=0.1,
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        random_state=42
    ))
])

extra_trees = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=1
    ))
])

rfr = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=1
    ))
])

# XGBoost
xgb = Pipeline([
    ("transformer", preprocessor_tree),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=5.0,
        random_state=42,
        n_jobs=1,
        verbosity=0
    ))
])


# LightGBM
lgbm = Pipeline([
    ("transformer", preprocessor_tree),
    ("model", LGBMClassifier(
        objective="binary",
        n_estimators=200,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=15,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=5.0,
        random_state=42,
        n_jobs=1,
        verbosity=-1
    ))
])


# CatBoost
catboost = Pipeline([
    ("transformer", preprocessor_tree),
    ("model", CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=300,
        learning_rate=0.03,
        depth=3,
        l2_leaf_reg=5.0,
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=1
    ))
])

In [23]:
class RepeatedStratifiedGroupKFold:

    def __init__(
        self,
        n_splits=5,
        n_repeats=10,
        random_state=42
    ):
        self.n_splits = n_splits
        self.n_repeats = n_repeats
        self.random_state = random_state

    def split(self, X, y, groups):
        groups = np.asarray(groups)
        y = np.asarray(y)

        for repeat in range(self.n_repeats):
            cv = StratifiedGroupKFold(
                n_splits=self.n_splits,
                shuffle=True,
                random_state=self.random_state + repeat
            )

            for fold, (train_idx, val_idx) in enumerate(
                cv.split(X, y, groups),
                start=1
            ):
                assert set(groups[train_idx]).isdisjoint(
                    set(groups[val_idx])
                )

                yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits * self.n_repeats

In [28]:
groups_pred_train = base.loc[
    X_pred_train.index,
    LUMA_CLUSTER
]

groups_pred_test = base.loc[
    X_pred_test.index,
    LUMA_CLUSTER
]

In [27]:
TABLES = Path("tables")
TABLES.mkdir(parents=True, exist_ok=True)

FIGURES = TABLES / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

In [38]:
models = {
    "LogisticRegression": lr,
    "ExtraTreesClassifier": extra_trees,
    "RandomForestClassifier": rfr,
    "XGBClassifier": xgb,
    "LGBMClassifier": lgbm,
    "CatBoostClassifier": catboost
}

cv = RepeatedStratifiedGroupKFold(
    n_splits=5,
    n_repeats=2,
    random_state=42
)


cv_results = []
test_results = []

test_probabilities = {}
test_predictions = {}

mlflow.set_experiment(
    "Model Experiments and Testing Set Evaluation"
)

for name, model in models.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    with mlflow.start_run(run_name=name):

        # ----------------------------------------------------
        # CROSS-VALIDATION
        # ----------------------------------------------------

        cv_scores = cross_validate(
            model,
            X_pred_train,
            y_pred_train,
            groups=groups_pred_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision",
                "balanced_accuracy": "balanced_accuracy",
                "f1": "f1",
                "precision": "precision",
                "recall": "recall"
            },
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        cv_summary = {}

        for metric in [
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "precision",
            "recall"
        ]:

            scores = cv_scores[f"test_{metric}"]

            cv_summary[f"CV_{metric}_mean"] = np.mean(scores)
            cv_summary[f"CV_{metric}_std"] = np.std(scores)

            mlflow.log_metric(
                f"CV_{metric}_mean",
                float(np.mean(scores))
            )

            mlflow.log_metric(
                f"CV_{metric}_std",
                float(np.std(scores))
            )

        cv_results.append({
            "Model": name,
            **cv_summary
        })

        # ----------------------------------------------------
        # FINAL TRAINING
        # ----------------------------------------------------

        model.fit(
            X_pred_train,
            y_pred_train
        )

        # ----------------------------------------------------
        # TEST PREDICTION
        # ----------------------------------------------------

        y_pred = model.predict(X_pred_test)
        y_prob = model.predict_proba(X_pred_test)[:, 1]

        test_predictions[name] = y_pred
        test_probabilities[name] = y_prob

        # ----------------------------------------------------
        # TEST METRICS
        # ----------------------------------------------------

        metrics = {
            "Accuracy": accuracy_score(y_pred_test, y_pred),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_pred_test, y_pred
            ),
            "Precision": precision_score(
                y_pred_test, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_pred_test, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_pred_test, y_pred, zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_pred_test, y_prob
            ),
            "PR_AUC": average_precision_score(
                y_pred_test, y_prob
            ),
            "Brier_Score": brier_score_loss(
                y_pred_test, y_prob
            )
        }

        test_results.append({
            "Model": name,
            **metrics
        })

        for metric, value in metrics.items():
            mlflow.log_metric(metric, float(value))

        # ----------------------------------------------------
        # CONFUSION MATRIX
        # ----------------------------------------------------

        cm = confusion_matrix(
            y_pred_test,
            y_pred,
            labels=[0, 1]
        )

        fig, ax = plt.subplots(figsize=(6, 5))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Not Recovered", "Recovered"],
            yticklabels=["Not Recovered", "Recovered"],
            ax=ax
        )

        ax.set_title(f"{name} - Test Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

        plt.tight_layout()

        cm_path = FIGURES / f"{name}_confusion_matrix.png"

        fig.savefig(
            cm_path,
            dpi=300,
            bbox_inches="tight"
        )

        mlflow.log_artifact(str(cm_path))
        plt.close(fig)

        # ----------------------------------------------------
        # CLASSIFICATION REPORT
        # ----------------------------------------------------

        report = classification_report(
            y_pred_test,
            y_pred,
            labels=[0, 1],
            target_names=["Not Recovered", "Recovered"],
            output_dict=True,
            zero_division=0
        )

        pd.DataFrame(report).T.to_csv(
            TABLES / f"{name}_classification_report.csv"
        )

        # ----------------------------------------------------
        # MODEL LOGGING
        # ----------------------------------------------------

        mlflow.sklearn.log_model(
            model,
            artifact_path="model"
        )

        print("CV ROC-AUC:", round(cv_summary["CV_roc_auc_mean"], 4))
        print("Test ROC-AUC:", round(metrics["ROC_AUC"], 4))
        print("Test PR-AUC:", round(metrics["PR_AUC"], 4))
        print("Test F1:", round(metrics["F1"], 4))

cv_df = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)

cv_df.to_csv(
    TABLES / "cross_validation_results.csv",
    index=False
)

test_df.to_csv(
    TABLES / "test_set_evaluation.csv",
    index=False
)

print("\nCROSS-VALIDATION RESULTS")
print(cv_df.round(4).to_string(index=False))

print("\nTEST SET RESULTS")
print(
    test_df.sort_values(
        "ROC_AUC",
        ascending=False
    ).round(4).to_string(index=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, probabilities in test_probabilities.items():

    RocCurveDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=axes[0]
    )

    PrecisionRecallDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=axes[1]
    )

axes[0].set_title("ROC Curves - Test Set")
axes[1].set_title("Precision-Recall Curves - Test Set")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_roc_pr_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

probability_df = pd.DataFrame(test_probabilities)

prediction_corr = probability_df.corr(method="spearman")

prediction_corr.to_csv(
    TABLES / "model_probability_correlation.csv"
)

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    prediction_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=ax
)

ax.set_title("Model Prediction Correlation - Test Set")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_probability_correlation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

best_model_name = (
    cv_df.sort_values(
        "CV_roc_auc_mean",
        ascending=False
    ).iloc[0]["Model"]
)

best_model = models[best_model_name]

print("Learning curve model:", best_model_name)

train_sizes, train_scores, validation_scores = learning_curve(
    best_model,
    X_pred_train,
    y_pred_train,
    groups=groups_pred_train,
    cv=cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.3, 1.0, 5),
    n_jobs=1,
    shuffle=True,
    random_state=42
)

train_mean = np.nanmean(train_scores, axis=1)
train_std = np.nanstd(train_scores, axis=1)

validation_mean = np.nanmean(validation_scores, axis=1)
validation_std = np.nanstd(validation_scores, axis=1)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training ROC-AUC"
)

ax.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15
)

ax.plot(
    train_sizes,
    validation_mean,
    marker="o",
    label="Validation ROC-AUC"
)

ax.fill_between(
    train_sizes,
    validation_mean - validation_std,
    validation_mean + validation_std,
    alpha=0.15
)

ax.set_title(f"Learning Curve - {best_model_name}")
ax.set_xlabel("Number of Training Observations")
ax.set_ylabel("ROC-AUC")
ax.legend()

plt.tight_layout()

fig.savefig(
    FIGURES / "best_model_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

prediction_df = pd.DataFrame(test_predictions)

prediction_df["Actual"] = y_pred_test.to_numpy()

prediction_df["Disagreement"] = (
    prediction_df.drop(columns="Actual")
    .nunique(axis=1)
)

prediction_df.to_csv(
    TABLES / "model_prediction_disagreement.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(8, 5))

prediction_df["Disagreement"].value_counts().sort_index().plot(
    kind="bar",
    ax=ax
)

ax.set_title("Model Prediction Disagreement")
ax.set_xlabel("Number of Distinct Predicted Classes")
ax.set_ylabel("Number of Test Observations")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_prediction_disagreement.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)


LogisticRegression


2026/11/20 01:20:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 01:20:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


CV ROC-AUC: 0.6914
Test ROC-AUC: 0.6461
Test PR-AUC: 0.5501
Test F1: 0.5581

ExtraTreesClassifier


2026/11/20 01:20:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 01:20:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


CV ROC-AUC: 0.6755
Test ROC-AUC: 0.6878
Test PR-AUC: 0.6097
Test F1: 0.6087

RandomForestClassifier


2026/11/20 01:21:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 01:21:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


... (3 outputs left out)

2026/11/20 01:21:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 01:21:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


CV ROC-AUC: 0.6662
Test ROC-AUC: 0.637
Test PR-AUC: 0.5259
Test F1: 0.5833

CatBoostClassifier


2026/11/20 01:21:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/20 01:21:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


CV ROC-AUC: 0.6311
Test ROC-AUC: 0.6062
Test PR-AUC: 0.4959
Test F1: 0.6122

CROSS-VALIDATION RESULTS
                 Model  CV_roc_auc_mean  CV_roc_auc_std  CV_pr_auc_mean  CV_pr_auc_std  CV_balanced_accuracy_mean  CV_balanced_accuracy_std  CV_f1_mean  CV_f1_std  CV_precision_mean  CV_precision_std  CV_recall_mean  CV_recall_std
    LogisticRegression           0.6914          0.0981          0.6949         0.1302                     0.6543                    0.0842      0.6392     0.1044             0.6375            0.1153          0.6578         0.1343
  ExtraTreesClassifier           0.6755          0.0876          0.6464         0.1242                     0.6225                    0.0886      0.5953     0.1362             0.5990            0.1263          0.6247         0.1970
RandomForestClassifier           0.6869          0.0884          0.6556         0.1058                     0.6569                    0.1049      0.6393     0.1398             0.6315            0.1218      

## Proposed Ensemble

In [40]:
class JumpProcessExplainabilityEnsemble(ClassifierMixin, BaseEstimator):

    def __init__(
        self,
        estimators,
        weights=None,
        threshold=0.5,
    ):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        if not X.columns.is_unique:
            raise ValueError("Feature names must be unique.")

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        self.classes_ = np.unique(y)

        if len(self.classes_) != 2:
            raise ValueError("Binary classification is required.")

        if self.weights is None:
            weights = np.ones(len(self.estimators))
        else:
            weights = np.asarray(self.weights, dtype=float)

        if len(weights) != len(self.estimators):
            raise ValueError("One weight is required per estimator.")

        if np.any(weights < 0) or weights.sum() <= 0:
            raise ValueError("Weights must be nonnegative and nonzero.")

        self.weights_ = weights / weights.sum()
        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(X, y)
            self.estimators_.append((name, fitted))

        return self

    def predict_proba(self, X):
        check_is_fitted(self, "estimators_")

        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame.")

        X = X.loc[:, self.feature_names_in_]

        probabilities = []

        for _, estimator in self.estimators_:
            probabilities.append(estimator.predict_proba(X))

        probabilities = np.asarray(probabilities)

        return np.average(
            probabilities,
            axis=0,
            weights=self.weights_
        )

    def predict(self, X):
        probabilities = self.predict_proba(X)
        positive_idx = np.where(self.classes_ == 1)[0][0]

        return np.where(
            probabilities[:, positive_idx] >= self.threshold,
            1,
            0
        )

    def explain(self, X, y, scoring="roc_auc", n_repeats=20,
                random_state=42):
        """
        Held-out permutation importance.
        This measures predictive contribution, not causal effect.
        """
        check_is_fitted(self, "estimators_")

        result = permutation_importance(
            self,
            X,
            y,
            scoring=scoring,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=1
        )

        return (
            pd.DataFrame({
                "feature": X.columns,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std
            })
            .sort_values("importance_mean", ascending=False)
            .reset_index(drop=True)
        )


In [46]:
jump_train = engineer_jump_features(base.loc[X_base_train.index])
jump_test = engineer_jump_features(base.loc[X_base_test.index])

X_jump_train = X_base_train.join(jump_train)
X_jump_test = X_base_test.join(jump_test)

assert X_jump_train.columns.equals(X_jump_test.columns)
assert X_jump_train.index.equals(X_base_train.index)
assert X_jump_test.index.equals(X_base_test.index)

print("Training shape:", X_jump_train.shape)
print("Testing shape:", X_jump_test.shape)
print(X_jump_train[JUMP_FEATURES].sum())

Training shape: (192, 22)
Testing shape: (48, 22)
processing_delay_jump       0
sensor_variability_jump     3
molecular_signal_A_jump     0
molecular_signal_B_jump     0
fluorescence_signal_jump    0
response_latency_jump       0
dtype: int64


In [48]:
jpee = JumpProcessExplainabilityEnsemble(
    estimators=[
        ("logistic", lr),
        ("extra_trees", extra_trees),
        ("random_forest", rfr),
    ],
    weights=[1, 1, 1],
    threshold=0.5
)

In [51]:
models = {
    "LogisticRegression": lr,
    "ExtraTreesClassifier": extra_trees,
    "RandomForestClassifier": rfr,
    "XGBClassifier": xgb,
    "LGBMClassifier": lgbm,
    "CatBoostClassifier": catboost,
    "JumpProcessExplainabilityEnsemble": jpee
}

# Keep feature spaces explicit.
model_data = {
    name: (
        X_jump_train,
        X_jump_test
    ) if name == "JumpProcessExplainabilityEnsemble" else (
        X_base_train,
        X_base_test
    )
    for name in models
}

cv = RepeatedStratifiedGroupKFold(
    n_splits=5,
    n_repeats=2,
    random_state=42
)

cv_results = []
test_results = []

test_probabilities = {}
test_predictions = {}

mlflow.set_experiment(
    "LUMA-1 Jump Process Ensemble Experiments"
)


# ============================================================
# MODEL EVALUATION
# ============================================================

for name, model in models.items():

    X_train, X_test = model_data[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    with mlflow.start_run(run_name=name):

        mlflow.set_tag("experiment", "LUMA-1")
        mlflow.set_tag(
            "feature_representation",
            "jump_augmented" if name == "JumpProcessExplainabilityEnsemble"
            else "conventional"
        )

        # ----------------------------------------------------
        # CROSS-VALIDATION
        # ----------------------------------------------------

        cv_scores = cross_validate(
            model,
            X_train,
            y_pred_train,
            groups=groups_pred_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision",
                "balanced_accuracy": "balanced_accuracy",
                "f1": "f1",
                "precision": "precision",
                "recall": "recall"
            },
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        cv_summary = {}

        for metric in [
            "roc_auc",
            "pr_auc",
            "balanced_accuracy",
            "f1",
            "precision",
            "recall"
        ]:

            scores = cv_scores[f"test_{metric}"]

            mean_score = float(np.mean(scores))
            std_score = float(np.std(scores))

            cv_summary[f"CV_{metric}_mean"] = mean_score
            cv_summary[f"CV_{metric}_std"] = std_score

            mlflow.log_metric(
                f"CV_{metric}_mean",
                mean_score
            )

            mlflow.log_metric(
                f"CV_{metric}_std",
                std_score
            )

        cv_results.append({
            "Model": name,
            **cv_summary
        })

        # ----------------------------------------------------
        # FINAL TRAINING
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_pred_train
        )

        # ----------------------------------------------------
        # TEST PREDICTION
        # ----------------------------------------------------

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        test_predictions[name] = y_pred
        test_probabilities[name] = y_prob

        # ----------------------------------------------------
        # TEST METRICS
        # ----------------------------------------------------

        metrics = {
            "Accuracy": accuracy_score(y_pred_test, y_pred),

            "Balanced_Accuracy": balanced_accuracy_score(
                y_pred_test, y_pred
            ),

            "Precision": precision_score(
                y_pred_test, y_pred, zero_division=0
            ),

            "Recall": recall_score(
                y_pred_test, y_pred, zero_division=0
            ),

            "F1": f1_score(
                y_pred_test, y_pred, zero_division=0
            ),

            "ROC_AUC": roc_auc_score(
                y_pred_test, y_prob
            ),

            "PR_AUC": average_precision_score(
                y_pred_test, y_prob
            ),

            "Brier_Score": brier_score_loss(
                y_pred_test, y_prob
            )
        }

        test_results.append({
            "Model": name,
            **metrics
        })

        for metric, value in metrics.items():
            mlflow.log_metric(
                metric,
                float(value)
            )

        # ----------------------------------------------------
        # CONFUSION MATRIX
        # ----------------------------------------------------

        cm = confusion_matrix(
            y_pred_test,
            y_pred,
            labels=[0, 1]
        )

        fig, ax = plt.subplots(figsize=(6, 5))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Not Recovered", "Recovered"],
            yticklabels=["Not Recovered", "Recovered"],
            ax=ax
        )

        ax.set_title(f"{name} - Test Confusion Matrix")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

        plt.tight_layout()

        cm_path = FIGURES / f"{name}_confusion_matrix.png"

        fig.savefig(
            cm_path,
            dpi=300,
            bbox_inches="tight"
        )

        mlflow.log_artifact(str(cm_path))
        plt.close(fig)

        # ----------------------------------------------------
        # CLASSIFICATION REPORT
        # ----------------------------------------------------

        report = classification_report(
            y_pred_test,
            y_pred,
            labels=[0, 1],
            target_names=["Not Recovered", "Recovered"],
            output_dict=True,
            zero_division=0
        )

        report_df = pd.DataFrame(report).T

        report_path = TABLES / f"{name}_classification_report.csv"

        report_df.to_csv(report_path)

        mlflow.log_artifact(str(report_path))

        # ----------------------------------------------------
        # JPEE EXPLAINABILITY
        # ----------------------------------------------------

        if name == "JumpProcessExplainabilityEnsemble":

            explanation = model.explain(
                X_test,
                y_pred_test,
                scoring="roc_auc",
                n_repeats=30,
                random_state=42
            )

            explanation_path = (
                TABLES / "JPEE_permutation_importance.csv"
            )

            explanation.to_csv(
                explanation_path,
                index=False
            )

            mlflow.log_artifact(str(explanation_path))

            fig, ax = plt.subplots(figsize=(10, 7))

            plot_data = explanation.sort_values(
                "importance_mean",
                ascending=True
            )

            ax.barh(
                plot_data["feature"],
                plot_data["importance_mean"],
                xerr=plot_data["importance_std"]
            )

            ax.set_title(
                "JPEE - Held-out Permutation Importance"
            )

            ax.set_xlabel("Decrease in ROC-AUC")

            plt.tight_layout()

            importance_path = (
                FIGURES / "JPEE_permutation_importance.png"
            )

            fig.savefig(
                importance_path,
                dpi=300,
                bbox_inches="tight"
            )

            mlflow.log_artifact(str(importance_path))
            plt.close(fig)

        # ----------------------------------------------------
        # MODEL LOGGING
        # ----------------------------------------------------

        signature = infer_signature(
            X_train.head(5),
            model.predict(X_train.head(5))
        )

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=X_train.head(5)
        )

        print(
            "CV ROC-AUC:",
            round(cv_summary["CV_roc_auc_mean"], 4)
        )

        print(
            "Test ROC-AUC:",
            round(metrics["ROC_AUC"], 4)
        )

        print(
            "Test PR-AUC:",
            round(metrics["PR_AUC"], 4)
        )

        print(
            "Test F1:",
            round(metrics["F1"], 4)
        )


# ============================================================
# SAVE AGGREGATED RESULTS
# ============================================================

cv_df = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)

cv_df.to_csv(
    TABLES / "cross_validation_results.csv",
    index=False
)

test_df.to_csv(
    TABLES / "test_set_evaluation.csv",
    index=False
)

print("\nCROSS-VALIDATION RESULTS")
print(cv_df.round(4).to_string(index=False))

print("\nTEST SET RESULTS")
print(
    test_df.sort_values(
        "ROC_AUC",
        ascending=False
    ).round(4).to_string(index=False)
)


# ============================================================
# ROC AND PRECISION-RECALL COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, probabilities in test_probabilities.items():

    RocCurveDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=axes[0]
    )

    PrecisionRecallDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=axes[1]
    )

axes[0].set_title("ROC Curves - Test Set")
axes[1].set_title("Precision-Recall Curves - Test Set")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_roc_pr_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# MODEL PROBABILITY CORRELATION
# ============================================================

probability_df = pd.DataFrame(test_probabilities)

prediction_corr = probability_df.corr(
    method="spearman"
)

prediction_corr.to_csv(
    TABLES / "model_probability_correlation.csv"
)

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    prediction_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=ax
)

ax.set_title("Model Prediction Correlation - Test Set")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_probability_correlation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# LEARNING CURVE
# ============================================================

best_model_name = (
    cv_df.sort_values(
        "CV_roc_auc_mean",
        ascending=False
    ).iloc[0]["Model"]
)

best_model = models[best_model_name]

X_best_train, _ = model_data[best_model_name]

print("Learning curve model:", best_model_name)

train_sizes, train_scores, validation_scores = learning_curve(
    best_model,
    X_best_train,
    y_pred_train,
    groups=groups_pred_train,
    cv=cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.3, 1.0, 5),
    n_jobs=1,
    shuffle=True,
    random_state=42
)

train_mean = np.nanmean(train_scores, axis=1)
train_std = np.nanstd(train_scores, axis=1)

validation_mean = np.nanmean(validation_scores, axis=1)
validation_std = np.nanstd(validation_scores, axis=1)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training ROC-AUC"
)

ax.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15
)

ax.plot(
    train_sizes,
    validation_mean,
    marker="o",
    label="Validation ROC-AUC"
)

ax.fill_between(
    train_sizes,
    validation_mean - validation_std,
    validation_mean + validation_std,
    alpha=0.15
)

ax.set_title(f"Learning Curve - {best_model_name}")
ax.set_xlabel("Number of Training Observations")
ax.set_ylabel("ROC-AUC")
ax.legend()

plt.tight_layout()

fig.savefig(
    FIGURES / "best_model_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# MODEL PREDICTION DISAGREEMENT
# ============================================================

prediction_df = pd.DataFrame(test_predictions)

prediction_df["Actual"] = np.asarray(y_pred_test)

prediction_df["Disagreement"] = (
    prediction_df.drop(columns="Actual")
    .nunique(axis=1)
)

prediction_df.to_csv(
    TABLES / "model_prediction_disagreement.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(8, 5))

prediction_df["Disagreement"].value_counts().sort_index().plot(
    kind="bar",
    ax=ax
)

ax.set_title("Model Prediction Disagreement")
ax.set_xlabel("Number of Distinct Predicted Classes")
ax.set_ylabel("Number of Test Observations")

plt.tight_layout()

fig.savefig(
    FIGURES / "model_prediction_disagreement.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("\nJPEE EVALUATION COMPLETE.")



LogisticRegression
CV ROC-AUC: 0.6914
Test ROC-AUC: 0.6461
Test PR-AUC: 0.5501
Test F1: 0.5581

ExtraTreesClassifier
CV ROC-AUC: 0.6755
Test ROC-AUC: 0.6878
Test PR-AUC: 0.6097
Test F1: 0.6087

RandomForestClassifier
CV ROC-AUC: 0.6869
Test ROC-AUC: 0.6788
Test PR-AUC: 0.5731
Test F1: 0.625

XGBClassifier
CV ROC-AUC: 0.6707
Test ROC-AUC: 0.6552
Test PR-AUC: 0.5486
Test F1: 0.625

LGBMClassifier
CV ROC-AUC: 0.6662
Test ROC-AUC: 0.637
Test PR-AUC: 0.5259
Test F1: 0.5833

CatBoostClassifier
CV ROC-AUC: 0.6311
Test ROC-AUC: 0.6062
Test PR-AUC: 0.4959
Test F1: 0.6122

JumpProcessExplainabilityEnsemble
CV ROC-AUC: 0.6958
Test ROC-AUC: 0.677
Test PR-AUC: 0.628
Test F1: 0.6383

CROSS-VALIDATION RESULTS
                            Model  CV_roc_auc_mean  CV_roc_auc_std  CV_pr_auc_mean  CV_pr_auc_std  CV_balanced_accuracy_mean  CV_balanced_accuracy_std  CV_f1_mean  CV_f1_std  CV_precision_mean  CV_precision_std  CV_recall_mean  CV_recall_std
               LogisticRegression           0.6914   

## Test set Evaluation

In [59]:
ROOT = Path("/home/voltsy/Projects/DataLensSubmission")

DATA = (
    ROOT / "data/processed/external/daphnia/"
    "daphnia_luma_schema.csv"
)

OUTPUT = ROOT / "results/daphnia"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SEED = 42



daphnia = pd.read_csv(DATA)

TARGET = "recovery_state_binary"

FEATURES = [
    "treatment_group",
    "recovery_temperature_c",
    "body_mass"
]

NUMERIC_FEATURES = [
    "recovery_temperature_c",
    "body_mass"
]

CATEGORICAL_FEATURES = [
    "treatment_group"
]

X = daphnia[FEATURES].copy()
y = daphnia[TARGET].astype(int).copy()

assert not X.isna().all().any()
assert y.nunique() == 2

# ------------------------------------------------------------
# STRATIFIED HELD-OUT SPLIT
# ------------------------------------------------------------

X_pred_train, X_pred_test, y_pred_train, y_pred_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=SEED
    )
)

# Same reduced feature representation for all models.
X_base_train = X_pred_train.copy()
X_base_test = X_pred_test.copy()

# No documented Daphnia measurement-order sequence exists.
# Therefore, no fabricated jump features.
X_jump_train = X_base_train.copy()
X_jump_test = X_base_test.copy()

# Five folds, two repeats, stratified.
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=SEED
)

print("=" * 70)
print("DAPHNIA MODELING DATA")
print("=" * 70)

print("Full dataset:", X.shape)
print("Training:", X_pred_train.shape)
print("Testing:", X_pred_test.shape)

print("\nTraining target:")
print(y_pred_train.value_counts().sort_index())

print("\nTesting target:")
print(y_pred_test.value_counts().sort_index())

print("\nFeatures:", FEATURES)


DAPHNIA MODELING DATA
Full dataset: (321, 3)
Training: (256, 3)
Testing: (65, 3)

Training target:
recovery_state_binary
0    124
1    132
Name: count, dtype: int64

Testing target:
recovery_state_binary
0    31
1    34
Name: count, dtype: int64

Features: ['treatment_group', 'recovery_temperature_c', 'body_mass']


In [62]:
ROOT = "/home/voltsy/Projects/DataLensSubmission"

DATA_PATH = os.path.join(
    ROOT,
    "data/processed/external/daphnia/daphnia_luma_schema.csv"
)

OUTPUT_DIR = os.path.join(
    ROOT,
    "results/daphnia"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

NUMERIC_FEATURES = [
    "recovery_temperature_c",
    "body_mass"
]

CATEGORICAL_FEATURES = [
    "treatment_group"
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "recovery_state_binary"



print("=" * 75)
print("DAPHNIA SHARED-FEATURE BENCHMARK")
print("=" * 75)

df = pd.read_csv(DATA_PATH)

if TARGET not in df.columns:
    if "recovery_state" not in df.columns:
        raise KeyError(
            "Neither recovery_state_binary nor recovery_state exists."
        )

    df[TARGET] = (
        pd.to_numeric(df["recovery_state"], errors="coerce") > 0
    ).astype(int)

required_columns = FEATURES + [TARGET]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = df[required_columns].copy()

for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="coerce"
)

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

if not set(df[TARGET].unique()).issubset({0, 1}):
    raise ValueError("Target must contain only binary values 0 and 1.")

X = df[FEATURES].copy()
y = df[TARGET].copy()



X_pred_train, X_pred_test, y_pred_train, y_pred_test = (
    train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )
)

X_base_train = X_pred_train.copy()
X_base_test = X_pred_test.copy()

# No documented equivalent of LUMA's measurement-order
# jump indicators exists in the Daphnia dataset.
#
# JPEE therefore receives the same three shared predictors.
# This is a reduced-feature JPEE, not the LUMA jump-augmented version.

X_jump_train = X_base_train.copy()
X_jump_test = X_base_test.copy()



def make_preprocessor(numeric_transformer):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "transformer",
            numeric_transformer
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


lr_preprocessor = make_preprocessor(
    StandardScaler()
)

tree_preprocessor = make_preprocessor(
    QuantileTransformer(
        n_quantiles=100,
        output_distribution="normal",
        subsample=10000,
        random_state=RANDOM_STATE
    )
)


lr = Pipeline([
    ("preprocessor", lr_preprocessor),
    (
        "model",
        __import__(
            "sklearn.linear_model",
            fromlist=["LogisticRegression"]
        ).LogisticRegression(
            C=1.0,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

extra_trees = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        __import__(
            "sklearn.ensemble",
            fromlist=["ExtraTreesClassifier"]
        ).ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features=0.9,
            class_weight="balanced",
            n_jobs=1,
            random_state=RANDOM_STATE
        )
    )
])

rfr = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        __import__(
            "sklearn.ensemble",
            fromlist=["RandomForestClassifier"]
        ).RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features=0.9,
            class_weight="balanced",
            n_jobs=1,
            random_state=RANDOM_STATE
        )
    )
])

xgb = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            eval_metric="logloss",
            n_jobs=1,
            random_state=RANDOM_STATE,
            verbosity=0
        )
    )
])

lgbm = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        LGBMClassifier(
            n_estimators=250,
            max_depth=3,
            num_leaves=7,
            learning_rate=0.03,
            reg_lambda=2.0,
            verbosity=-1,
            n_jobs=1,
            random_state=RANDOM_STATE
        )
    )
])

catboost = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        CatBoostClassifier(
            iterations=250,
            depth=4,
            learning_rate=0.03,
            l2_leaf_reg=3.0,
            verbose=False,
            random_seed=RANDOM_STATE,
            allow_writing_files=False,
            thread_count=1
        )
    )
])



class JumpProcessExplainabilityEnsemble(
    ClassifierMixin,
    BaseEstimator
):

    def __init__(
        self,
        estimators,
        weights=None,
        threshold=0.5
    ):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold

    def fit(self, X, y):

        self.estimators_ = []

        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(X, y)
            self.estimators_.append((name, fitted))

        self.classes_ = np.unique(y)

        if len(self.classes_) != 2:
            raise ValueError(
                "JPEE requires a binary classification target."
            )

        if self.weights is None:
            weights = np.ones(len(self.estimators_))

        else:
            weights = np.asarray(
                self.weights,
                dtype=float
            )

        if len(weights) != len(self.estimators_):
            raise ValueError(
                "Number of weights must match number of estimators."
            )

        if np.any(weights < 0) or weights.sum() == 0:
            raise ValueError(
                "Weights must be nonnegative and sum to a positive value."
            )

        self.weights_ = weights / weights.sum()

        return self

    def predict_proba(self, X):

        probabilities = []

        for _, estimator in self.estimators_:
            probabilities.append(
                estimator.predict_proba(X)[:, 1]
            )

        probabilities = np.asarray(probabilities)

        positive_probability = np.average(
            probabilities,
            axis=0,
            weights=self.weights_
        )

        return np.column_stack([
            1 - positive_probability,
            positive_probability
        ])

    def predict(self, X):

        probabilities = self.predict_proba(X)[:, 1]

        return (
            probabilities >= self.threshold
        ).astype(int)

    def explain(
        self,
        X,
        y,
        scoring="roc_auc",
        n_repeats=30,
        random_state=42
    ):

        result = permutation_importance(
            self,
            X,
            y,
            scoring=scoring,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=1
        )

        return (
            pd.DataFrame({
                "feature": X.columns,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std
            })
            .sort_values(
                "importance_mean",
                ascending=False
            )
            .reset_index(drop=True)
        )


jpee = JumpProcessExplainabilityEnsemble(
    estimators=[
        ("logistic", lr),
        ("extra_trees", extra_trees),
        ("random_forest", rfr)
    ],
    weights=[1, 1, 1],
    threshold=0.5
)



models = {
    "LogisticRegression": lr,
    "ExtraTreesClassifier": extra_trees,
    "RandomForestClassifier": rfr,
    "XGBClassifier": xgb,
    "LGBMClassifier": lgbm,
    "CatBoostClassifier": catboost,
    "JumpProcessExplainabilityEnsemble": jpee
}

model_data = {
    name: (
        X_jump_train,
        X_jump_test
    ) if name == "JumpProcessExplainabilityEnsemble" else (
        X_base_train,
        X_base_test
    )
    for name in models
}


cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=RANDOM_STATE
)



mlflow.set_experiment(
    "Daphnia Shared Feature Domain Shift - JPEE"
)

cv_results = []
test_results = []

test_probabilities = {}
test_predictions = {}



scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

for name, model in models.items():

    X_train, X_test = model_data[name]

    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    with mlflow.start_run(run_name=name):

        mlflow.set_tag("experiment", "Daphnia")
        mlflow.set_tag(
            "dataset",
            "Frizot et al. 2026 - Daphnia"
        )
        mlflow.set_tag(
            "feature_representation",
            "shared_three_feature_reduced_jpee"
            if name == "JumpProcessExplainabilityEnsemble"
            else "shared_three_feature_conventional"
        )
        mlflow.set_tag(
            "target",
            "recovery_state_binary"
        )
        mlflow.set_tag(
            "validation",
            "RepeatedStratifiedKFold"
        )

        # ----------------------------------------------------
        # CROSS-VALIDATION
        # ----------------------------------------------------

        cv_scores = cross_validate(
            model,
            X_train,
            y_pred_train,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=1,
            error_score="raise"
        )

        cv_row = {"Model": name}

        for metric in scoring:

            test_key = f"test_{metric}"
            train_key = f"train_{metric}"

            mean_score = np.mean(
                cv_scores[test_key]
            )

            std_score = np.std(
                cv_scores[test_key]
            )

            train_mean = np.mean(
                cv_scores[train_key]
            )

            cv_row[f"{metric}_mean"] = mean_score
            cv_row[f"{metric}_std"] = std_score
            cv_row[f"{metric}_train_mean"] = train_mean

            mlflow.log_metric(
                f"cv_{metric}_mean",
                float(mean_score)
            )

            mlflow.log_metric(
                f"cv_{metric}_std",
                float(std_score)
            )

            mlflow.log_metric(
                f"train_{metric}_mean",
                float(train_mean)
            )

            print(
                f"{metric}: "
                f"{mean_score:.4f} ± {std_score:.4f}"
            )

        cv_results.append(cv_row)

        # ----------------------------------------------------
        # FINAL TRAINING
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_pred_train
        )

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        test_predictions[name] = y_pred
        test_probabilities[name] = y_prob

        # ----------------------------------------------------
        # TEST METRICS
        # ----------------------------------------------------

        metrics = {
            "Accuracy": accuracy_score(
                y_pred_test,
                y_pred
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_pred_test,
                y_pred
            ),
            "Precision": precision_score(
                y_pred_test,
                y_pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_pred_test,
                y_pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_pred_test,
                y_pred,
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_pred_test,
                y_prob
            ),
            "PR_AUC": average_precision_score(
                y_pred_test,
                y_prob
            ),
            "Brier_Score": brier_score_loss(
                y_pred_test,
                y_prob
            )
        }

        test_row = {
            "Model": name,
            **metrics
        }

        test_results.append(test_row)

        for metric, value in metrics.items():

            mlflow.log_metric(
                f"test_{metric.lower()}",
                float(value)
            )

            print(
                f"{metric}: {value:.4f}"
            )

        # ----------------------------------------------------
        # CONFUSION MATRIX
        # ----------------------------------------------------

        cm = confusion_matrix(
            y_pred_test,
            y_pred
        )

        cm_df = pd.DataFrame(
            cm,
            index=["Actual_0", "Actual_1"],
            columns=["Predicted_0", "Predicted_1"]
        )

        cm_path = os.path.join(
            OUTPUT_DIR,
            f"{name}_confusion_matrix.csv"
        )

        cm_df.to_csv(cm_path)

        mlflow.log_artifact(cm_path)

        # ----------------------------------------------------
        # CLASSIFICATION REPORT
        # ----------------------------------------------------

        report = classification_report(
            y_pred_test,
            y_pred,
            output_dict=True,
            zero_division=0
        )

        report_path = os.path.join(
            OUTPUT_DIR,
            f"{name}_classification_report.json"
        )

        with open(report_path, "w") as file:
            json.dump(
                report,
                file,
                indent=4
            )

        mlflow.log_artifact(report_path)

        # ----------------------------------------------------
        # JPEE EXPLAINABILITY
        # ----------------------------------------------------

        if name == "JumpProcessExplainabilityEnsemble":

            explanation = model.explain(
                X_test,
                y_pred_test,
                scoring="roc_auc",
                n_repeats=30,
                random_state=RANDOM_STATE
            )

            explanation_path = os.path.join(
                OUTPUT_DIR,
                "JPEE_feature_explainability.csv"
            )

            explanation.to_csv(
                explanation_path,
                index=False
            )

            mlflow.log_artifact(
                explanation_path
            )

            print("\n[JPEE FEATURE EXPLAINABILITY]")
            print(explanation)

        # ----------------------------------------------------
        # MODEL LOGGING
        # ----------------------------------------------------

        from mlflow.models import infer_signature

        signature = infer_signature(
            X_train.head(5),
            model.predict(X_train.head(5))
        )

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=X_train.head(5)
        )



cv_df = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)

cv_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_cv_results.csv"
)

test_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_test_results.csv"
)

cv_df.to_csv(cv_path, index=False)
test_df.to_csv(test_path, index=False)

with mlflow.start_run(run_name="Daphnia_Aggregated_Results"):

    mlflow.log_artifact(cv_path)
    mlflow.log_artifact(test_path)

print("\n" + "=" * 75)
print("CROSS-VALIDATION RESULTS")
print("=" * 75)

print(
    cv_df.sort_values(
        "roc_auc_mean",
        ascending=False
    ).to_string(index=False)
)

print("\n" + "=" * 75)
print("HELD-OUT TEST RESULTS")
print("=" * 75)

print(
    test_df.sort_values(
        "ROC_AUC",
        ascending=False
    ).to_string(index=False)
)


fig, ax = plt.subplots(figsize=(9, 7))

for name, probabilities in test_probabilities.items():

    RocCurveDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=ax
    )

ax.set_title("Daphnia - ROC Curves")
ax.grid(alpha=0.3)

roc_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_roc_curves.png"
)

fig.savefig(
    roc_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))

for name, probabilities in test_probabilities.items():

    PrecisionRecallDisplay.from_predictions(
        y_pred_test,
        probabilities,
        name=name,
        ax=ax
    )

ax.set_title("Daphnia - Precision-Recall Curves")
ax.grid(alpha=0.3)

pr_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_pr_curves.png"
)

fig.savefig(
    pr_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)



probability_df = pd.DataFrame(
    test_probabilities
)

probability_correlation = probability_df.corr(
    method="spearman"
)

correlation_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_probability_correlation.csv"
)

probability_correlation.to_csv(
    correlation_path
)

fig, ax = plt.subplots(figsize=(9, 7))

image = ax.imshow(
    probability_correlation,
    aspect="auto"
)

ax.set_xticks(
    range(len(probability_correlation.columns))
)

ax.set_yticks(
    range(len(probability_correlation.index))
)

ax.set_xticklabels(
    probability_correlation.columns,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    probability_correlation.index
)

ax.set_title(
    "Daphnia - Spearman Probability Correlation"
)

fig.colorbar(image, ax=ax)

correlation_plot_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_probability_correlation.png"
)

fig.savefig(
    correlation_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)



best_model_name = (
    cv_df.sort_values(
        "roc_auc_mean",
        ascending=False
    )
    .iloc[0]["Model"]
)

best_model = models[best_model_name]

X_best_train, _ = model_data[best_model_name]

print("\nBest model by mean CV ROC-AUC:", best_model_name)

train_sizes, train_scores, validation_scores = learning_curve(
    best_model,
    X_best_train,
    y_pred_train,
    cv=cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.3, 1.0, 5),
    n_jobs=1,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_mean = np.mean(
    train_scores,
    axis=1
)

train_std = np.std(
    train_scores,
    axis=1
)

validation_mean = np.mean(
    validation_scores,
    axis=1
)

validation_std = np.std(
    validation_scores,
    axis=1
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    train_sizes,
    train_mean,
    marker="o",
    label="Training ROC-AUC"
)

ax.plot(
    train_sizes,
    validation_mean,
    marker="o",
    label="Validation ROC-AUC"
)

ax.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.15
)

ax.fill_between(
    train_sizes,
    validation_mean - validation_std,
    validation_mean + validation_std,
    alpha=0.15
)

ax.set_xlabel("Training Samples")
ax.set_ylabel("ROC-AUC")
ax.set_title(
    f"Learning Curve - {best_model_name}"
)

ax.legend()
ax.grid(alpha=0.3)

learning_curve_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_learning_curve.png"
)

fig.savefig(
    learning_curve_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)




prediction_df = pd.DataFrame(
    test_predictions
)

disagreement = pd.DataFrame(
    index=prediction_df.columns,
    columns=prediction_df.columns,
    dtype=float
)

for model_a in prediction_df.columns:

    for model_b in prediction_df.columns:

        disagreement.loc[model_a, model_b] = np.mean(
            prediction_df[model_a] != prediction_df[model_b]
        )

disagreement_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_prediction_disagreement.csv"
)

disagreement.to_csv(
    disagreement_path
)

fig, ax = plt.subplots(figsize=(9, 7))

image = ax.imshow(
    disagreement.astype(float),
    aspect="auto"
)

ax.set_xticks(
    range(len(disagreement.columns))
)

ax.set_yticks(
    range(len(disagreement.index))
)

ax.set_xticklabels(
    disagreement.columns,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    disagreement.index
)

ax.set_title(
    "Daphnia - Pairwise Prediction Disagreement"
)

fig.colorbar(image, ax=ax)

disagreement_plot_path = os.path.join(
    OUTPUT_DIR,
    "daphnia_prediction_disagreement.png"
)

fig.savefig(
    disagreement_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)


with mlflow.start_run(
    run_name="Daphnia_Global_Analysis"
):

    for path in [
        roc_path,
        pr_path,
        correlation_path,
        correlation_plot_path,
        learning_curve_path,
        disagreement_path,
        disagreement_plot_path
    ]:

        mlflow.log_artifact(path)


print("\n" + "=" * 75)
print("DAPHNIA BENCHMARK COMPLETE")
print("=" * 75)

print("Dataset:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
print("Training samples:", len(X_pred_train))
print("Testing samples:", len(X_pred_test))
print("Features:", FEATURES)
print("Target:", TARGET)
print("Best CV model:", best_model_name)

print("\nGenerated files:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

print("\nMLflow experiment:")
print("Daphnia Shared Feature Domain Shift - JPEE")


DAPHNIA SHARED-FEATURE BENCHMARK

[DATA AUDIT]
Shape: (321, 4)
Features: ['treatment_group', 'recovery_temperature_c', 'body_mass']
Target: recovery_state_binary

Target distribution:
recovery_state_binary
0    155
1    166
Name: count, dtype: int64

Target proportions:
recovery_state_binary
0    0.482866
1    0.517134
Name: proportion, dtype: float64

Missing values:
treatment_group           0
recovery_temperature_c    0
body_mass                 0
dtype: int64

[DATA SPLIT]
Training: (256, 3)
Testing: (65, 3)
Training target:
recovery_state_binary
0    124
1    132
Name: count, dtype: int64
Testing target:
recovery_state_binary
0    31
1    34
Name: count, dtype: int64

LogisticRegression
roc_auc: 0.7784 ± 0.0476
pr_auc: 0.8215 ± 0.0365
balanced_accuracy: 0.7153 ± 0.0497
f1: 0.7156 ± 0.0508
precision: 0.7368 ± 0.0524
recall: 0.6969 ± 0.0586
Accuracy: 0.7231
Balanced_Accuracy: 0.7296
Precision: 0.8333
Recall: 0.5882
F1: 0.6897
ROC_AUC: 0.8022
PR_AUC: 0.8249
Brier_Score: 0.1875

Extra